In [1]:
# import necessary packages
import xarray as xr
import numpy as np
import pandas as pd
import os
import glob

In [2]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# function to create the season category general
short_shifts = [0.5, 1.5]
medium_shifts = [2.5, 3.5]
long_shifts = [4.5, 5.5, 6.5]
def make_season_dict(months, season_prefix, short_shifts, medium_shifts, long_shifts):
    short = [months[0] - s for s in short_shifts]
    medium = [months[0] - s for s in medium_shifts]
    long = [months[0] - s for s in long_shifts]
    return {
        f'{season_prefix}_short': short,
        f'{season_prefix}_medium': medium,
        f'{season_prefix}_long': long,
        'months': months
    }

In [4]:
# use this function when Dec is the start
short_shifts_DJF = [0.5, 1.5, 12.5, 13.5]
medium_shifts_DJF = [2.5, 3.5, 14.5, 15.5]
long_shifts_DJF = [4.5, 5.5, 6.5, 16.5, 17.5, 18.5]
def make_season_dict_DJF(months, season_prefix, short_shifts_DJF, medium_shifts_DJF, long_shifts_DJF):
    short = [months[0] - s for s in short_shifts_DJF]
    medium = [months[0] - s for s in medium_shifts_DJF]
    long = [months[0] - s for s in long_shifts_DJF]
    return {
        f'{season_prefix}_short': short,
        f'{season_prefix}_medium': medium,
        f'{season_prefix}_long': long,
        'months': months
    }


In [5]:
regions_seasons_dict = {
    'eastern_east_africa': {
        'OND': make_season_dict([10, 11, 12], 'OND', short_shifts, medium_shifts, long_shifts),
        'MAM': make_season_dict([3, 4, 5], 'MAM', short_shifts, medium_shifts, long_shifts)
    },
    'lake_victoria_basin': {
        'DJF': make_season_dict_DJF([12, 1, 2], 'DJF', short_shifts_DJF, medium_shifts_DJF, long_shifts_DJF),
        'MAM': make_season_dict([3, 4, 5], 'MAM', short_shifts, medium_shifts, long_shifts),
        'SON': make_season_dict([9, 10, 11], 'SON', short_shifts, medium_shifts, long_shifts)
    },
    'west_africa': {
        'JAS': make_season_dict([7, 8, 9], 'JAS', short_shifts, medium_shifts, long_shifts)
    },
    'southern_africa': {
        'DJF': make_season_dict_DJF([12, 1, 2], 'DJF', short_shifts_DJF, medium_shifts_DJF, long_shifts_DJF),
        'FMA': make_season_dict([2, 3, 4], 'FMA', short_shifts, medium_shifts, long_shifts)
    },
    'south_sudan': {
        'MJJ':make_season_dict([5, 6, 7], 'MJJ', short_shifts, medium_shifts, long_shifts),
        'JAS':make_season_dict([7, 8, 9], 'JAS', short_shifts, medium_shifts, long_shifts),
        'ASO':make_season_dict([8, 9, 10], 'ASO', short_shifts, medium_shifts, long_shifts)
    },
    'eastern_ukraine': {
        'DJF':make_season_dict_DJF([12, 1, 2], 'DJF', short_shifts_DJF, medium_shifts_DJF, long_shifts_DJF),
        'AMJ':make_season_dict([4, 5, 6], 'AMJ', short_shifts, medium_shifts, long_shifts),
        'JA':make_season_dict([7, 8], 'JA', short_shifts, medium_shifts, long_shifts),
    },
    'sri_lanka': {
        'OND':make_season_dict([10, 11, 12], 'OND', short_shifts, medium_shifts, long_shifts)
    }
}


In [6]:
season_names_dict = {
    'south_sudan': ['MJJ', 'JAS', 'ASO'],
    'eastern_east_africa': ['OND', 'MAM'],
    'lake_victoria_basin': ['DJF', 'MAM', 'SON'],
    'west_africa': ['JAS'],
    'southern_africa': ['DJF', 'FMA'],
    'south_sudan': ['MJJ', 'JAS', 'ASO'],
    'eastern_ukraine': ['DJF', 'AMJ', 'JA'],
    'sri_lanka': ['OND']
}

In [7]:
def convert_monthly_to_seasonal(file_path, regions_seasons_dict, season_names_dict, save_path):
  """
  This function takes a merged monthly netcdf file and converts it to seasonal

  Arguments:
  - file_path: path to merged monthly netcdf file
  - regions_seasons_dict: dictionary of regions and their seasons
  - save_path: path to save the seasonal netcdf file

  Usage Notes:
  Ensure that the file_path follows this format:
  '/content/drive/MyDrive/data/netCDF/eastern_east_africa_CanESM5_merged.nc'
  It does not matter what the netcdf file name is, as long as it follows the format of
  some_region_here_model_merged.nc

  Ensure that the save_path follows this format:
  '/content/drive/MyDrive/data/netCDF'
  Again, it does not matter what the folder name is, as long as it does not end with a '/' or anything else after the folder name.

  Data Notes:
  The merged monthly netcdf file used have the following columns:
  - latitude
  - longitude
  - predicted_precip
  - actual_precip (from CHIRPS)
  - date
  - lead_time

  This merged data has been pre-processed with the monthly_merged_data_generation.py script.

  Regions and Seasons Notes:
  The regions_seasons_dict is a dictionary of regions, their seasons, and specific values of month minus lead time.
  Please refer to the above matrix of month minus lead time values to understand how these values were determined,
  as well as how the dictionary works.
  """

  # extract relevant information from the file path
  split = file_path.split('/') # split into list

  file_name = split[-1] # get the file name

  name_split = file_name.split('_') # get the name of the region, i.e [eastern, east, africa]

  region_name = '_'.join(name_split[0:-2]) # combine the name of the region, i.e eastern_east_africa

  new_file_name= file_name.replace('.nc', '_seasonal.csv') # make new file name for saving

  # status
  print(f'Converting {file_name} to seasonal...')

  # check if region name is in regions_seasons_dict
  if region_name not in regions_seasons_dict:
      print(f"ValueError: Region '{region_name}' not found in regions_seasons_dict.")
      return

  # open file
  merged_monthly_file = xr.open_dataset(file_path)

  # convert to a dataframe for pre-processing
  merged_monthly_file_df = merged_monthly_file.to_dataframe().reset_index().dropna().groupby(['time', 'lead_time', 'latitude', 'longitude'])[['predicted_precip', 'precip']].mean().reset_index()

  # seperate month and year into seperate columns
  merged_monthly_file_df['month'] = merged_monthly_file_df['time'].dt.month
  merged_monthly_file_df['year'] = merged_monthly_file_df['time'].dt.year

  # create month_minus_lead_time
  merged_monthly_file_df['month_minus_lead_time'] = merged_monthly_file_df['month'] - merged_monthly_file_df['lead_time']

  # access the dictionary items of the given region
  season_data = regions_seasons_dict[region_name]

  # Iterate through seasons in the region
  for season_name, season_dict in season_data.items():
      months = season_dict['months']

      # Filter only the rows for the relevant season months, i.e only OND months
      seasonal_df = merged_monthly_file_df[merged_monthly_file_df['month'].isin(months)].copy()

      # Create a new column for the current season
      column_name = f"{season_name}"
      merged_monthly_file_df[column_name] = None  # initialize empty season column

      # Iterate over each lead category (i.e OND_short)
      for lead_label, lead_values in season_dict.items():
          if lead_label != 'months':
              # Find matching rows based on month_minus_lead_time
              matching_idx = seasonal_df[seasonal_df['month_minus_lead_time'].isin(lead_values)].index

              # Assign the lead_label to the season column in original dataframe
              merged_monthly_file_df.loc[matching_idx, column_name] = lead_label

  # drop uneccesary column
  merged_monthly_file_df.drop(['month_minus_lead_time'], axis=1, inplace=True)

  seasons = season_names_dict[region_name] # get the list of seasons for that region

  season_dfs = [] # initialzie empty list of seasonal dataframes

  # for each season in season list, pivot the data into separate dataframes
  # if a region has 3 seasons,there will be 3 dataframes
  for season in seasons:
      if season in merged_monthly_file_df.columns:
          temp_df = merged_monthly_file_df[['time', 'lead_time', 'latitude', 'longitude',
                                'predicted_precip', 'precip', 'year', 'month', season]].copy()
          temp_df = temp_df.rename(columns={season: 'lead_category'})
          temp_df = temp_df.dropna(subset=['lead_category'])  # Drop rows where lead_category is NaN
          temp_df['season'] = season
          season_dfs.append(temp_df)

  # Combine all seasonal rows
  long_format_df = pd.concat(season_dfs, ignore_index=True)

  # Keep only the last part after splitting by underscore (i.e, OND_short -> short)
  long_format_df['lead_category'] = long_format_df['lead_category'].str.split('_').str[-1]

  # extract current model from file path
  current_model = name_split[-2]

  # assign model name column
  long_format_df['model'] = str(current_model)
  long_format_df['region'] = str(region_name)

  # save as csv
  new_save_path = save_path + '/' + new_file_name
  long_format_df.to_csv(new_save_path)

  return long_format_df

In [8]:
test_df = convert_monthly_to_seasonal('/content/drive/MyDrive/capstone_data/netCDF/eastern_east_africa_CanESM5_merged.nc',
                                      regions_seasons_dict=regions_seasons_dict, season_names_dict=season_names_dict,
                                      save_path='/content/drive/MyDrive/capstone_data/csv/seasonal')

Converting eastern_east_africa_CanESM5_merged.nc to seasonal...


In [9]:
test_df

,time,lead_time,latitude,longitude,predicted_precip,precip,year,month,lead_category,season,model,region
0,1991-10-01,0.5,-3.5,38.0,0.854860,29.213007,1991,10,short,OND,CanESM5,eastern_east_africa
1,1991-10-01,0.5,-3.5,38.5,0.854860,21.352221,1991,10,short,OND,CanESM5,eastern_east_africa
2,1991-10-01,0.5,-3.5,39.0,0.979578,32.822708,1991,10,short,OND,CanESM5,eastern_east_africa
3,1991-10-01,0.5,-3.5,39.5,0.979578,109.046631,1991,10,short,OND,CanESM5,eastern_east_africa
4,1991-10-01,0.5,-3.0,38.0,1.064233,12.410067,1991,10,short,OND,CanESM5,eastern_east_africa
...,...,...,...,...,...,...,...,...,...,...,...,...
465547,2021-05-01,8.5,8.0,47.5,2.273299,68.415924,2021,5,long,MAM,CanESM5,eastern_east_africa
465548,2021-05-01,8.5,8.0,48.0,2.131540,42.353031,2021,5,long,MAM,CanESM5,eastern_east_africa
465549,2021-05-01,8.5,8.0,48.5,2.131540,83.079788,2021,5,long,MAM,CanESM5,eastern_east_africa
465550,2021-05-01,8.5,8.0,49.0,1.718496,80.556221,2021,5,long,MAM,CanESM5,eastern_east_africa


In [ ]:
# use glob to get all file names in netCDF
file_paths = glob.glob('/content/drive/MyDrive/capstone_data/netCDF/*.nc')

for file_name in file_paths:
    convert_monthly_to_seasonal(file_name,
                                      regions_seasons_dict=regions_seasons_dict, season_names_dict=season_names_dict,
                                      save_path='/content/drive/MyDrive/capstone_data/csv/seasonal')

Converting west_africa_CMCC_merged.nc to seasonal...
Converting southern_africa_CMCC_merged.nc to seasonal...
Converting eastern_ukraine_CMCC_merged.nc to seasonal...
Converting eastern_east_africa_CMCC_merged.nc to seasonal...
Converting south_sudan_CMCC_merged.nc to seasonal...
Converting lake_victoria_basin_ECMWF_merged.nc to seasonal...
Converting sri_lanka_ECMWF_merged.nc to seasonal...
Converting west_africa_ECMWF_merged.nc to seasonal...
Converting southern_africa_ECMWF_merged.nc to seasonal...
Converting eastern_ukraine_ECMWF_merged.nc to seasonal...
Converting eastern_east_africa_ECMWF_merged.nc to seasonal...
Converting south_sudan_ECMWF_merged.nc to seasonal...
Converting lake_victoria_basin_DWD_merged.nc to seasonal...
Converting sri_lanka_DWD_merged.nc to seasonal...
Converting west_africa_DWD_merged.nc to seasonal...
Converting southern_africa_DWD_merged.nc to seasonal...
Converting eastern_ukraine_DWD_merged.nc to seasonal...
Converting eastern_east_africa_DWD_merged.nc 